# Day 7 Lab 05: Bronze Schema Drift

        **Layer:** Bronze  
        **Python reference:** `Lab_Files/labs/lab_05_bronze_schema_drift.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read the Bronze order table.
- Profile every column for type and null counts.
- Use sparse optional columns as schema-drift signals.

**Dependency:** Run Lab/Notebook 04 first so `lake/bronze/orders_raw` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:
from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import OUTPUT_DIR, ensure_output_dirs, read_bronze_orders, require_source_data, spark_session, write_csv_dir, schema_profile

## 2. Start Spark and Read Bronze

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook05BronzeSchemaDrift")

bronze = read_bronze_orders(spark)
print("Bronze table loaded for profiling.")

Bronze table loaded for profiling.


## 3. Inspect Bronze Schema

In [4]:
bronze.printSchema()
bronze.show(10, truncate=False)

root
 |-- _bronze_ingested_at: string (nullable = true)
 |-- _ingestion_batch_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- delivery_promise: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- status: string (nullable = true)

+--------------------------+----------------------+--------------------------+-------+-------+-----------+--------+-----------+----------------+--------+--------------------+--------+----------+---------+
|_bronze_ingested_at       |_ingestion_batch_id   |_source_file              |amount |channel|coupon_code|currency|customer_id|delivery_promise|event_id|event_time          |order_id|product_id|

## 4. Build Column Profile Rows

In [5]:
profile = schema_profile(bronze)
profile.show(50, truncate=False)

total_rows = bronze.count()
profile_column_count = len(bronze.schema.fields)
print(f"Profiled {profile_column_count} columns with Spark-native profiling.")

+-------------------+---------+-------------+---------+
|column_name        |data_type|non_null_rows|null_rows|
+-------------------+---------+-------------+---------+
|_bronze_ingested_at|string   |12           |0        |
|_ingestion_batch_id|string   |12           |0        |
|_source_file       |string   |12           |0        |
|amount             |double   |12           |0        |
|channel            |string   |12           |0        |
|coupon_code        |string   |2            |10       |
|currency           |string   |12           |0        |
|customer_id        |bigint   |12           |0        |
|delivery_promise   |string   |2            |10       |
|event_id           |string   |12           |0        |
|event_time         |string   |12           |0        |
|order_id           |bigint   |12           |0        |
|product_id         |string   |11           |1        |
|status             |string   |12           |0        |
+-------------------+---------+-------------+---

## 5. Create and Display Schema Profile

In [6]:
profile.orderBy("column_name").show(50, truncate=False)

+-------------------+---------+-------------+---------+
|column_name        |data_type|non_null_rows|null_rows|
+-------------------+---------+-------------+---------+
|_bronze_ingested_at|string   |12           |0        |
|_ingestion_batch_id|string   |12           |0        |
|_source_file       |string   |12           |0        |
|amount             |double   |12           |0        |
|channel            |string   |12           |0        |
|coupon_code        |string   |2            |10       |
|currency           |string   |12           |0        |
|customer_id        |bigint   |12           |0        |
|delivery_promise   |string   |2            |10       |
|event_id           |string   |12           |0        |
|event_time         |string   |12           |0        |
|order_id           |bigint   |12           |0        |
|product_id         |string   |11           |1        |
|status             |string   |12           |0        |
+-------------------+---------+-------------+---

## 6. Write Profile Output

In [12]:
write_csv_dir(profile, OUTPUT_DIR / "notebook_05_bronze_schema_profile")
print(f"Profiled {profile_column_count} Bronze columns across {total_rows} rows.")

Py4JJavaError: An error occurred while calling o347.csv.
: java.util.NoSuchElementException: None.get
	at scala.None$.get(Option.scala:529)
	at scala.None$.get(Option.scala:527)
	at org.apache.spark.sql.execution.datasources.BasicWriteJobStatsTracker$.metrics(BasicWriteStatsTracker.scala:242)
	at org.apache.spark.sql.execution.command.DataWritingCommand.metrics(DataWritingCommand.scala:55)
	at org.apache.spark.sql.execution.command.DataWritingCommand.metrics$(DataWritingCommand.scala:55)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.metrics$lzycompute(InsertIntoHadoopFsRelationCommand.scala:47)
	at org.apache.spark.sql.execution.datasources.InsertIntoHadoopFsRelationCommand.metrics(InsertIntoHadoopFsRelationCommand.scala:47)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.metrics$lzycompute(commands.scala:109)
	at org.apache.spark.sql.execution.command.DataWritingCommandExec.metrics(commands.scala:109)
	at org.apache.spark.

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [8]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()